# Day 1 — Exploratory Data Analysis

**Goal:** Understand the dataset before building anything.

We answer 5 questions:
1. What does the data look like?
2. Are there any missing values?
3. How sparse is the user-item matrix?
4. What does the rating distribution look like?
5. Who are the most active users and most rated movies?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('../assets', exist_ok=True)

In [3]:
DATA_PATH = '../data/'

movies_df  = pd.read_csv(DATA_PATH + 'movies.csv')
ratings_df = pd.read_csv(DATA_PATH + 'ratings.csv')

print(f'Movies  shape: {movies_df.shape}')
print(f'Ratings shape: {ratings_df.shape}')

Movies  shape: (62423, 3)
Ratings shape: (25000095, 4)


In [4]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [6]:
print('=== Movies missing values ===')
print(movies_df.isnull().sum())

print('\n=== Ratings missing values ===')
print(ratings_df.isnull().sum())

=== Movies missing values ===
movieId    0
title      0
genres     0
dtype: int64

=== Ratings missing values ===
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [7]:
n_users   = ratings_df['userId'].nunique()
n_movies  = ratings_df['movieId'].nunique()
n_ratings = len(ratings_df)

total_possible = n_users * n_movies
sparsity = (1 - n_ratings / total_possible) * 100

print(f'Users:           {n_users:,}')
print(f'Movies:          {n_movies:,}')
print(f'Ratings:         {n_ratings:,}')
print(f'Total possible:  {total_possible:,}')
print(f'Sparsity:        {sparsity:.2f}%')

Users:           162,541
Movies:          59,047
Ratings:         25,000,095
Total possible:  9,597,558,427
Sparsity:        99.74%


In [8]:
min_movies_rating = 50
min_user_rating = 20

active_users = ratings_df['userId'].value_counts()
active_users = active_users[active_users >= min_user_rating].index

active_movies = ratings_df['movieId'].value_counts()
active_movies = active_movies[active_movies >= min_movies_rating].index

filtered_ratings = ratings_df[ratings_df['userId'].isin(active_users) & ratings_df['movieId'].isin(active_movies)]

In [9]:
n_users_f   = filtered_ratings['userId'].nunique()
n_movies_f  = filtered_ratings['movieId'].nunique()
n_ratings_f = len(filtered_ratings)

sparsity_f = (1 - n_ratings_f / (n_users_f * n_movies_f)) * 100

print(f'Filtered users:   {n_users_f:,}')
print(f'Filtered movies:  {n_movies_f:,}')
print(f'Filtered ratings: {n_ratings_f:,}')
print(f'Sparsity:         {sparsity_f:.2f}%')

Filtered users:   162,540
Filtered movies:  13,176
Filtered ratings: 24,644,928
Sparsity:         98.85%


In [13]:
merged_df = pd.merge(movies_df, filtered_ratings, on='movieId')

top_movies = (
    merged_df.groupby('title')['rating'].agg(n_ratings='count', avg_rating='mean').sort_values('n_ratings', ascending=False).head(10).round(2)
)

print('Top 10 most rated movies:')
print(top_movies)

Top 10 most rated movies:
                                           n_ratings  avg_rating
title                                                           
Forrest Gump (1994)                            81491        4.05
Shawshank Redemption, The (1994)               81482        4.41
Pulp Fiction (1994)                            79672        4.19
Silence of the Lambs, The (1991)               74127        4.15
Matrix, The (1999)                             72674        4.15
Star Wars: Episode IV - A New Hope (1977)      68717        4.12
Jurassic Park (1993)                           64144        3.68
Schindler's List (1993)                        60411        4.25
Braveheart (1995)                              59184        4.00
Fight Club (1999)                              58773        4.23


In [12]:
filtered_ratings.to_parquet('../data/ratings_filtered.parquet', index=False)